# Compare wavelet power based on Ra-Ap firing rate in merged window

This notebook calculates an average spike triggered wavelet spectrogram for each spike type.

## Initialization - don't change

In [ ]:
import sys
sys.path.append('..')

from tqdm import tqdm

from workflows.run_util.io import collect_paths_from_table
from param_classes.workflows.wavelet_analysis import (
    WorkflowParams,
    WindowedWaveletAnalysisParams
)
from workflows.triggered_wavelet_analysis import run_analysis

params = WorkflowParams[WindowedWaveletAnalysisParams](
    analysis=WindowedWaveletAnalysisParams()
)

## Color settings

Set the colors that will be used for coloring the markers on the strip plot and the cellwise average firing frequency curve.

In [ ]:
params.palette = {
    'cell_type': {
        'PYR_L2-3': '#803300ff',
        'PYR_L4-6': '#502d16ff',
        'RSI_L2-3': '#ff48a3ff',
        'RSI_L4-6': '#800080ff',
        'FSI_L2-3': '#d54500ff',
        'FSI_L4-6': '#7a0000ff'
    },
    'spike_type': {
        'RA-AP': '#00ff00',
        'UNKNOWN': '#000000b2'
    }
}

## Data settings

Set raw input dir. You need to give the base path where the group directories can be found.

In [ ]:
MAT_DIR = 'C:/Datasets/raap/invivo'
PROJECT_DIR = 'C:/tmp/Bozso2025/PutRAAP/Results/New'

In [ ]:
params.data.raw_data.path = f'{MAT_DIR}'
params.data.raw_data.extension='mat'

Set the base path of the directory where csv files containing the spike features and categories can be found.
> It needs to be the same structure as the `raw_data` directory.

In [ ]:
params.data.spike_features.path = f'{PROJECT_DIR}/SpikeFeatures'
params.data.spike_features.extension = 'csv'

## Input-output settings

Set the Excel table containing the cell attributes, grouping and selector information.

In [ ]:
params.input = {
    'mode': 'table',
    'table_path': f'{PROJECT_DIR}/data_invivo.xlsx',
    'select_rules': [
        {'selected': 1}
    ]
}

Set the output base directory path. All results of this analysis (cellwise aggregated tables, grouped results, figures, statistics) is going to saved there.

In [ ]:
params.output.path = f'{PROJECT_DIR}/RaAp_Containing_Window_Power'

## Analysis parameters

 - `params.analysis.cut_window`: window size to cut, relative to the spike

In [ ]:
params.analysis.window_size = 1.0
params.analysis.step_size = 0.1

 - `params.analysis.signal.name`: signal name to analyze
 - `params.analysis.signal.fs`: resampled sampling rate of the signal

In [ ]:
params.analysis.signal.name = 'wideband'
params.analysis.signal.fs = 5000

 - `params.analysis.wavelet.low`: lowest analyzed frequency
 - `params.analysis.wavelet.high`: highest analyzed frequency
 - `params.analysis.wavelet.step`: step size between analyzed frequencies
 - `params.analysis.wavelet.n_cycles`: number of cycles in the wavelet
 - `params.analysis.wavelet.length`: length of the wavelet

In [ ]:
params.analysis.wavelet.low = 1.0
params.analysis.wavelet.high = 30.0
params.analysis.wavelet.step = 1.0
params.analysis.wavelet.n_cycles = 7
params.analysis.wavelet.length = 1.0

Mode to calculate the baseline power: `mean` or `median`

In [ ]:
normalization_mode = 'median'

In [ ]:
raap_threshold = 10

In [ ]:
n_quantile = 5
bin_size = 2

## Plot parameters

 - `params.analysis.spectrogram.clim`: ends of the colorbar

In [ ]:
params.analysis.clim = (None, None)
# params.analysis.clim = (0, 3e4)

## Initialize analysis

In [ ]:
from workflows.windowed_wavelet_analysis import (
    create_wavelet_saver,
    create_spectrogram_segment_extractor,
    create_recording_wavelet_calculator,
    create_windowed_segmenter
)
save_spectrogram = create_wavelet_saver(params.output.path)
extract_segments = create_spectrogram_segment_extractor(params.analysis)
calculate_spectrogram = create_recording_wavelet_calculator(params.analysis)
cut_segments = create_windowed_segmenter(params.analysis)

## Run analysis

Collect input files according to the table.

In [ ]:
input_files = collect_paths_from_table(params.data, params.input)

### Individual analysis

Perform individual analysis and save the results.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from workflows.windowed_wavelet_analysis import (
    aggregate_spectrograms,
    collect_spiking_segments,
    calculate_baseline_spectrogram_multi_spiketype,
    create_spectrogram_normalizer,
    normalize_spectrogram_segment,
    create_segment_power_matrix,
    plot_spectrogram_with_power_and_raster,
    segment_to_active_intervals,
    collect_spike_triggered_spectrograms,
    plot_spectrogram_with_power,
)

for cell in tqdm(input_files, total=len(input_files), desc="Processing cells"):
    cell_data = cell.data.load()
    cell_palette = {
        'AIS-AP': params.palette['cell_type'][cell.info.cell_type],
        'UNKNOWN': params.palette['spike_type']['UNKNOWN'],
        'RA-AP': params.palette['spike_type']['RA-AP']
    }

    spectrogram = calculate_spectrogram(cell_data)
    baseline_segments = collect_spike_triggered_spectrograms(
        spectrogram,
        cell_data.spikes,
        window_size=params.analysis.window_size,
        spike_type='AIS-AP',
        exclude_spike_types=['RA-AP', 'UNKNOWN']
    )
    raap_segments = collect_spike_triggered_spectrograms(
        spectrogram,
        cell_data.spikes,
        window_size=params.analysis.window_size,
        spike_type='RA-AP'
    )

    print(f"AIS-AP triggered segments without surrounding RA-AP or UNKNOWN: {len(baseline_segments)}")
    print(f"RA-AP triggered segments: {len(raap_segments)}")

    baseline_spectrogram = calculate_baseline_spectrogram_multi_spiketype(
        baseline_segments,
        exclude_types=['RA-AP', 'UNKNOWN']
    )
    normalize = create_spectrogram_normalizer(
        baseline_spectrogram, mode=normalization_mode
    )

    mean_raap_power = aggregate_spectrograms([s.spectrogram for s in raap_segments])

    raap_interval_segments = segment_to_active_intervals(
        spectrogram,
        cell_data.spikes,
        window_size=params.analysis.window_size,
        include_spike_types=['RA-AP']
    )

    raap_interval_beginnings = [
        segment.with_spectrogram(
            segment.spectrogram.cut_around(
                0, (0, 2*params.analysis.window_size)
            )
        )
        for segment in raap_interval_segments
    ]


    baseline_spectrogram = normalize(baseline_spectrogram)
    mean_raap_power = normalize(mean_raap_power)
    raap_interval_beginnings = [
        normalize_spectrogram_segment(segment, normalize=normalize)
        for segment in raap_interval_beginnings
    ]

    fig = plot_spectrogram_with_power(
        baseline_spectrogram,
        params.analysis.clim,
        fig_size=(5, 3)
    )
    fig.suptitle(
        f"{cell.info.name} - mean AIS-AP triggered power n={len(baseline_segments)}"
    )
    plt.show()

    fig = plot_spectrogram_with_power(
        mean_raap_power,
        params.analysis.clim,
        fig_size=(5, 3)
    )
    fig.suptitle(
        f"{cell.info.name} - mean RA-AP triggered power n={len(raap_segments)}"
    )
    plt.show()

    fig = plot_spectrogram_with_power_and_raster(
        raap_interval_beginnings,
        params.analysis.clim,
        fig_size=(5, 5),
        palette=cell_palette,
        raster_tick_size=2
    )
    fig.suptitle(
        f"{cell.info.name} - " +
        f"Ra-Ap segments" +
        f" (n={len(raap_interval_beginnings)})"
    )
    plt.show()

    for raap_interval_segment in raap_interval_segments:
        fig = plot_spectrogram_with_power_and_raster(
            [raap_interval_segment],
            params.analysis.clim,
            fig_size=(5, 5),
            palette=cell_palette,
            raster_tick_size=2
        )
        fig.suptitle(
            f"{cell.info.name} - " +
            f"t={raap_interval_segment.trigger:.3f}" +
            f" size={raap_interval_segment.spectrogram.times[-1] - raap_interval_segment.spectrogram.times[0]:.3f}s"
        )
        plt.show()

    break


In [ ]:
# firing_power_matrix = create_segment_power_matrix(normalized_segments, 'median')
# fig, ax = plt.subplots(figsize=(8, 6))
# im = ax.imshow(
#     firing_power_matrix.power,
#     aspect='auto',
#     extent=(
#         firing_power_matrix.x[0],
#         firing_power_matrix.x[-1],
#         firing_power_matrix.y[0],
#         firing_power_matrix.y[-1]
#     ),
#     origin='lower',
#     vmin=params.analysis.clim[0],
#     vmax=params.analysis.clim[1],
#     cmap='jet',
#     interpolation='bicubic'
# )

# cb = fig.colorbar(
#     im,
#     ax=ax,
#     pad=0.02,
#     fraction=0.05,
#     location='right',
#     anchor=(0, 0.5)
# )

# cb.set_label('Median power')
# ax.set_xlabel('Ra-Ap avg firing (Hz)')
# ax.set_ylabel('Frequency (Hz)')
# plt.show()